In [8]:
import pandas as pd

# Source

* Data Source:
    * Ministry of Land, Infrastructure and Transport (MOLIT). 2024. Building Permit Statistics. (`../resources/BPS-2024-MOLIT`)
    * Ministry of Land, Infrastructure and Transport (MOLIT). 2024. Building Stock Statistics. (`../resources/BSS-2024-MOLIT`)

* Implemented Input Files
    * `/input/policy/korea-2035/buildings/zeb_cp.xml`
    * `/input/policy/korea-2035/buildings/zeb_ep.xml`

# Zero Energy Building (ZEB) Regulation

South Korea’s ZEB regulation sets different requirements for public and private buildings based on floorspace and energy self-sufficiency rate (ESSR).  
ZEB is classified into grades:  
- **Grade 5**: ESSR ≥ 20%  
- **Grade 4**: ESSR ≥ 40%  
- **Grade 3**: ESSR ≥ 60%  

### Public Buildings
| Year | Floorspace Threshold | Required Grade | ESSR Requirement |
|------|----------------------|----------------|------------------|
| 2020 | ≥ 500 m²             | Grade 4        | ≥ 40%            |
| 2030+| ≥ 500 m²             | Grade 3        | ≥ 60%            |

### Private Buildings
| Year   | Floorspace Threshold | Required Grade | ESSR Requirement |
|--------|----------------------|----------------|------------------|
| Current| ≥ 1,000 m²           | Grade 5        | ≥ 20%            |

---

## Scenario Assumptions

- **Current Policy**: Regulations follow the official schedule above.  
- **Enhanced Policy**: From **2030**, Grade 4 (≥ 40% ESSR) is applied to new private buildings with a total floor area of **500 m² or more**.


We model the ZEB regulation in two steps:

1. **Equivalent full self-sufficiency floor space calculation**  
   For each building, we convert the achieved energy self-sufficiency rate (ESSR) into an equivalent floor area that would reach 100% self-sufficiency.  
   - Example: A building with 1,000 m² floor space and 30% ESSR is treated as equivalent to a 300 m² building with full self-sufficiency.

2. **Energy demand adjustment via shell conductance**  
   Using the unit energy consumption (EJ/m²), we estimate the total energy associated with the equivalent self-sufficient floor space.  
   Instead of explicitly supplying this energy via rooftop PV in GCAM, we adjust the **shell conductance** parameter for the relevant building technologies.  
   - **Role of shell conductance in GCAM**: Shell conductance represents the thermal transmittance of the building envelope (W/m²·K), affecting the rate of heat transfer between indoors and outdoors. Lower shell conductance values improve insulation, thereby reducing heating and cooling demand. By modifying this parameter, we effectively simulate the reduced net energy demand resulting from on-site renewable generation and improved building performance under ZEB regulations.

Detailed implementation steps are provided below.

We refer two external dataset `../resources/BPS-2024-MOLIT` and `../resources/BSS-2024-MOLIT` for estimation.

## Current Policies

In [1]:
tot_pr_floor_space = 4227660684.322 - 376645408.432
new_pr_floor_space = 147394196.42337 - 9195076.7802
new_pr_ratio = new_pr_floor_space / tot_pr_floor_space
new_pr_ratio

0.035886411697297435

In [2]:
fs_gt_1000_share = 0.07369
fs_gt_500_share = 0.15460
ss_ratio_g5 = 0.2
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio_g5 * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.002644469677973848 0.00819250892637603 0.013740548174778214 0.019288587423180396


In [3]:
fs_gt_500_share = 0.15460
ss_ratio = 0.2
private_vals = [fs_gt_1000_share * new_pr_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio]

In [4]:
tot_public_floor_space = 376645408.432
new_public_floor_space = 9195076.7802
new_public_ratio = new_public_floor_space / tot_public_floor_space
new_public_ratio

0.0244130860866716

In [5]:
ss_ratio_25 = 0.4
ss_ratio_30 = 0.6
new_public_ratio_ss_25 = fs_gt_500_share * new_public_ratio * ss_ratio_25 * 5
new_public_ratio_ss_30 = new_public_ratio_ss_25 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_35 = new_public_ratio_ss_30 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_40 = new_public_ratio_ss_35 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
print(new_public_ratio_ss_25, new_public_ratio_ss_30, new_public_ratio_ss_35, new_public_ratio_ss_40)

0.007548526217998858 0.018871315544997148 0.030194104871995437 0.04151689419899372


In [6]:
public_vals = [ss_ratio_25, ss_ratio_30, ss_ratio_30, ss_ratio_30]

In [9]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.003081
2030    0.009143
2035    0.015205
2040    0.021267
dtype: float64

## Enhanced Ambition

Private buildings are regulated by the same condition for the public buildings after 2030

In [10]:
tot_pr_floor_space = 4227660684.322 - 376645408.432
new_pr_floor_space = 147394196.42337 - 9195076.7802
new_pr_ratio = new_pr_floor_space / tot_pr_floor_space
new_pr_ratio

0.035886411697297435

In [11]:
fs_gt_1000_share = 0.07369
fs_gt_500_share = 0.15460
ss_ratio = 0.2
ss_ratio_30 = 0.4
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.002644469677973848 0.013740548174778214 0.024836626671582582 0.035932705168386946


In [12]:
private_vals += [fs_gt_1000_share * new_pr_ratio, fs_gt_500_share * ss_ratio_30, fs_gt_500_share * ss_ratio_30, fs_gt_500_share * ss_ratio_30]

In [13]:
tot_public_floor_space = 376645408.432
new_public_floor_space = 9195076.7802
new_public_ratio = new_public_floor_space / tot_public_floor_space
new_public_ratio

0.0244130860866716

In [14]:
ss_ratio_25 = 0.4
ss_ratio_30 = 0.6
new_public_ratio_ss_25 = fs_gt_500_share * new_public_ratio * ss_ratio_25 * 5
new_public_ratio_ss_30 = new_public_ratio_ss_25 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_35 = new_public_ratio_ss_30 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_40 = new_public_ratio_ss_35 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
print(new_public_ratio_ss_25, new_public_ratio_ss_30, new_public_ratio_ss_35, new_public_ratio_ss_40)

0.007548526217998858 0.018871315544997148 0.030194104871995437 0.04151689419899372


In [15]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.003081
2030    0.014197
2035    0.025313
2040    0.036430
dtype: float64